## Kane's method

When doing the lagrangian mechanics, we have already calculated $\dot{\vec{X}_w}$ and $\dot{\vec{X}_p}$, so we can save some trouble here.

### Kane's method intro

Basically:
$$F_r + F_r^* = 0$$

Where, $F_r$ is the generalized active force, and $F_r^*$ is the generalized inertia force.

The generalized active force is defined as:
$$
F_r = \sum_{i} \vec{F}_i \cdot \frac{\partial \dot{\vec{X}}_i}{\partial \dot{q}_r}
$$

Which in this case includes the gravity force and the control force for the two motors.

The generalized inertia force is defined as:
$$
F_r^* = -\sum_{i} m_i \ddot{\vec{X}}_i \cdot \frac{\partial \dot{\vec{X}}_i}{\partial \dot{q}_r}
$$ 

Intrinsically, the generalized inertia force is $\sum m_i {a}_i$.

Let $q = [q_1, q_2, q_3, q_4, q_5]^T = [\psi, \theta, \phi, r, \gamma]^T$ be the vector of generalized coordinates.

The wheel position is given by:
$$
\vec{X}_w = [X_G , Y_G, Z_G]^T
$$ 

Where:
$$
\begin{aligned}
\dot{x}_C &= \omega_1 R \sin \psi \cos \vartheta + \omega_2 R \cos \psi, \\
\dot{y}_C &= -\omega_1 R \cos \psi \cos \vartheta + \omega_2 R \sin \psi, \\
{z}_C &=  R \cos \vartheta \\
\omega_1 &:= \dot{\vartheta}, \\
\omega_2 &:= \dot{\varphi} + \dot{\psi} \sin \vartheta, \\
\omega_3 &:= \dot{\psi} \cos \vartheta, \\
\end{aligned}
$$

For the code, it's easier to calculate from $v_c = v_p + \omega \times r = \omega \times r$, where $v_p$ is the velocity of the pendulum center of mass, $\omega$ is the angular velocity of the wheel, and $r_{cw}$ is the vector from the wheel center to the pendulum center of mass.

Luckily we have python library that can do most of the work for us, with given $\vec{X}_i$ and $q_r$. As shown below:

In [ ]:
from sympy.physics.mechanics import ReferenceFrame, Point, dynamicsymbols, KanesMethod, Particle, RigidBody
from sympy import init_printing, simplify, symbols
from IPython.display import display

R, h, m, m_w, m_l, g = symbols('R h m m_w m_l g')

# Generalized coordinates and speeds
q1, q2, q3, q4, q5 = [dynamicsymbols(r'\psi'), dynamicsymbols(r'\theta'), dynamicsymbols(r'\phi'), dynamicsymbols('r'), dynamicsymbols(r'\gamma')]# psi, theta, phi, rho, gamma
u1, u2, u3, u4, u5 = [dynamicsymbols(r'\dot{\psi}'), dynamicsymbols(r'\dot{\theta}'), dynamicsymbols(r'\dot{\phi}'), dynamicsymbols(r'\dot{r}'), dynamicsymbols(r'\dot{\gamma}')]# corresponding generalized speeds

kde = {q1.diff(): u1, q2.diff(): u2, q3.diff(): u3, q4.diff(): u4, q5.diff(): u5} # type: ignore

E0 = ReferenceFrame('E0')
E1 = E0.orientnew('E1', 'Axis', [q1, E0.z]) # rotate around z-axis by psi
E2 = E1.orientnew('E2', 'Axis', [q2, E1.x]) # rotate around x-axis by theta
E3 = E2.orientnew('E3', 'Axis', [q5, E2.y]) # rotate around y-axis by gamma
EW = E3.orientnew('EW', 'Axis', [q3, E2.y]) # rotate around y-axis by phi, this is the frame attached to the wheel

O = Point('O') # origin
O.set_vel(E0, 0) # origin is fixed

################################### Wheel center kinematics ###################################

W = Point('W') # wheel center
W.set_pos(O, R*E2.z) # type: ignore     Initial position

w_total = E2.ang_vel_in(E0) + u3 * E2.y # Total angular velocity fo the wheel

r_cw = R * E2.z
v_w_vector = w_total.cross(r_cw)

W.set_vel(E0, v_w_vector) # type: ignore    

################################## Pendulum kinematics ########################################

P = W.locatenew('P', h * E3.z) # create Pendulum relative to the wheel
P.v2pt_theory(W, E0, E3)

################################## Linear motor kinematics ########################################

L = W.locatenew('L', (q4) * E2.y)
# L.set_vel(E2, u4 * E2.y) # relative velocity in E2 frame, which is the velocity of the linear motor relative to the wheel center
# L.v2pt_theory(W, E0, E2)

L.set_vel(E0, W.vel(E0) + u4 * E2.y + E2.ang_vel_in(E0).cross(L.pos_from(W))) # type: ignore     Absolute velocity of the linear motor, which is the velocity of the wheel center plus the relative velocity in E2 frame plus the contribution from the rotation of the E2 frame

############################# Define particles and forces ########################################
body_p = Particle('body_p', P, m)
body_w = Particle('body_w', W, m_w)
body_l = Particle('body_l', L, m_l)

#### Forces

T_W = dynamicsymbols('T_W') # torque applied to the wheel
F_L = dynamicsymbols('F_L') # force applied to the linear motor

forces = [
    (P, -m * g * E0.z),
    (W, -m_w * g * E0.z),
    (L, -m_l * g * E0.z),

    ## Wheel torque
    (E3, - T_W * E2.y),    # type: ignore  torque applied to the wheel, acting on the pendulum reversely
    (EW, T_W * E2.y),   # type: ignore  torque applied to the wheel
    ## Linear motor force
    (L, - F_L * E2.y),# type: ignore  
    (W, F_L * E2.y) # type: ignore  
]

########################### Summarize the system and derive equations of motion ############################

particles = [body_p, body_w, body_l]
q_list = [q1, q2, q3, q4, q5]
u_list = [u1, u2, u3, u4, u5]

kd_eqs = [u1 - q1.diff(), u2 - q2.diff(), u3 - q3.diff(), u4 - q4.diff(), u5 - q5.diff()] # type: ignore

kane = KanesMethod(E0, q_ind=q_list, u_ind=u_list, kd_eqs=kd_eqs) # on interial frame E0

(fr, frstar) = kane.kanes_equations(particles, forces) # Calculate the generalized active forces and inertia forces

mm = kane.mass_matrix_full
ff = kane.forcing_full


########################## Output ###########################

## M(q) * u' = f(q, u, t)

init_printing(use_latex='mathjax')
display(simplify(mm))
display(simplify(ff))

########################### Reorder and storage of equations of motion ############################

from sympy import lambdify
import numpy as np
import dill

args = [
    q1, q2, q3, q4, q5, 
    u1, u2, u3, u4, u5, 
    T_W, F_L, 
    R, h, m, m_w, m_l, g
]

mass_matrix_func = lambdify(args, mm, 'numpy')
forcing_func = lambdify(args, ff, 'numpy')

with open("system_dynamics.pkl", "wb") as f:
    dill.dump({'M': mass_matrix_func, 'f': forcing_func}, f)




⎡1  0  0  0  0                                                                 ↪
⎢                                                                              ↪
⎢0  1  0  0  0                                                                 ↪
⎢                                                                              ↪
⎢0  0  1  0  0                                                                 ↪
⎢                                                                              ↪
⎢0  0  0  1  0                                                                 ↪
⎢                                                                              ↪
⎢0  0  0  0  1                                                                 ↪
⎢                                                                              ↪
⎢                2        2                ⎛ 2    2                       2    ↪
⎢0  0  0  0  0  R ⋅m_w⋅sin (\theta(t)) + m⋅⎝R ⋅sin (\theta(t)) + 2⋅R⋅h⋅sin (\t ↪
⎢                           

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢   2                                                     2                    ↪
⎢- R ⋅m⋅\dot{\psi}(t)⋅\dot{\